# Protein Function Prediction – Data Preparation
**Project:** COMP 3608 B‑rank mission  
**This notebook:** downloads Kaggle datasets → `data/raw/`, cleans & merges them, engineers features, and saves processed data to `data/processed/`.

In [1]:
# 0. Environment
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
warnings.filterwarnings('ignore')

# Create directory structure
RAW_DIR = Path('data/raw')
PROC_DIR = Path('data/processed')
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# 1. Download datasets to data/raw/
import kagglehub

# GO annotations
path_go = kagglehub.dataset_download(
    "nikitamanaenkov/protein-sequences-with-go-annotations",
    output_dir=str(RAW_DIR / 'go_annotations')
)
# Simulated 1
path_sim1 = kagglehub.dataset_download(
    "willianoliveiragibin/bioinformatics-simulated",
    output_dir=str(RAW_DIR / 'simulated_1')
)
# Simulated 2
path_sim2 = kagglehub.dataset_download(
    "gallo33henrique/bioinformatics-protein-dataset-simulated",
    output_dir=str(RAW_DIR / 'simulated_2')
)
print("Datasets downloaded to:", str(RAW_DIR))

100%|██████████| 32.8M/32.8M [00:00<00:00, 49.0MB/s]

Extracting files...


100%|██████████| 2.52M/2.52M [00:00<00:00, 24.1MB/s]

Extracting files...


100%|██████████| 12.3M/12.3M [00:00<00:00, 73.3MB/s]

Extracting files...


Datasets downloaded to: data/raw


In [3]:
# 2. Load & explore
def find_csv(directory):
    return list(Path(directory).rglob('*.csv'))

df_go = pd.read_csv(find_csv(path_go)[0])
df_sim1 = pd.read_csv(find_csv(path_sim1)[0])
df_sim2 = pd.read_csv(find_csv(path_sim2)[0])

print("GO columns:", df_go.columns.tolist())
print("Sim1 columns:", df_sim1.columns.tolist())
print("Sim2 columns:", df_sim2.columns.tolist())

GO columns: ['Entry', 'Sequence', 'GO_list', 'GO', 'GO_id', 'GO_namespace', 'GO_parents', 'GO_children', 'seq_length', 'mol_weight', 'pI', 'gravy', 'instability', 'aromaticity', 'helix', 'turn', 'sheet', 'aa_A', 'aa_C', 'aa_D', 'aa_E', 'aa_F', 'aa_G', 'aa_H', 'aa_I', 'aa_K', 'aa_L', 'aa_M', 'aa_N', 'aa_P', 'aa_Q', 'aa_R', 'aa_S', 'aa_T', 'aa_V', 'aa_W', 'aa_Y']
Sim1 columns: ['ID_Proteína', 'Sequência', 'Massa_Molecular', 'Ponto_Isoelétrico', 'Hidrofobicidade', 'Carga_Total', 'Proporção_Polar', 'Proporção_Apolar', 'Comprimento_Sequência', 'Classe']
Sim2 columns: ['ID_Proteína', 'Sequência', 'Massa_Molecular', 'Ponto_Isoelétrico', 'Hidrofobicidade', 'Carga_Total', 'Proporção_Polar', 'Proporção_Apolar', 'Comprimento_Sequência', 'Classe']


In [4]:
# 3. Standardise columns to 'sequence' and 'label_raw'
def standardise_columns(df):
    """Auto‑detect and rename sequence + label columns."""
    df = df.copy()
    # Sequence column keywords (including Portuguese)
    seq_keywords = ['seq', 'sequência', 'sequencia']
    # Label column keywords (including Portuguese and GO-specific)
    label_keywords = ['go', 'label', 'function', 'class', 'classe']
    
    seq_cols = [c for c in df.columns if any(k in c.lower() for k in seq_keywords)]
    if not seq_cols:
        raise KeyError(f"No sequence column found in {list(df.columns)}")
    seq_col = seq_cols[0]
    
    label_cols = [c for c in df.columns if any(k in c.lower() for k in label_keywords)]
    if not label_cols:
        raise KeyError(f"No label column found in {list(df.columns)}")
    label_col = label_cols[0]
    
    print(f"Auto‑detected: seq='{seq_col}', label='{label_col}'")
    df.rename(columns={seq_col: 'sequence', label_col: 'label_raw'}, inplace=True)
    return df[['sequence', 'label_raw']]

df_go = standardise_columns(df_go)
df_sim1 = standardise_columns(df_sim1)
df_sim2 = standardise_columns(df_sim2)

print("\nSample GO:")
print(df_go.head(2))

Auto‑detected: seq='Sequence', label='GO_list'
Auto‑detected: seq='Sequência', label='Classe'
Auto‑detected: seq='Sequência', label='Classe'

Sample GO:
                                            sequence  \
0  MNIDMNWLGQLLGSDWEIFPAGGATGDAYYAKHNGQQLFLKRNSSP...   
1  MFKKHTISLLIIFLLASAVLAKPIEAHTVSPVNPNAQQTTKTVMNW...   

                                           label_raw  
0  ['cytoplasm [GO:0005737]', 'ATP binding [GO:00...  
1  ['extracellular region [GO:0005576]', 'mannan ...  


In [5]:
# 4. Map GO terms to high‑level functional categories
def map_go(go_str):
    if pd.isna(go_str): return 'other'
    s = str(go_str)
    if any(t in s for t in ['catalytic','hydrolase','kinase','transferase','enzyme']):
        return 'enzyme'
    if 'binding' in s or 'receptor' in s:
        return 'binding'
    if 'transporter' in s:
        return 'transporter'
    if 'signal' in s or 'transducer' in s:
        return 'signal_transduction'
    if 'structural' in s or 'cytoskeleton' in s:
        return 'structural'
    return 'other'

df_go['label'] = df_go['label_raw'].apply(map_go)

def clean_label(raw):
    return 'other' if pd.isna(raw) else str(raw).strip().lower().replace(' ', '_')

df_sim1['label'] = df_sim1['label_raw'].apply(clean_label)
df_sim2['label'] = df_sim2['label_raw'].apply(clean_label)

for d in [df_go, df_sim1, df_sim2]:
    d.drop(columns='label_raw', inplace=True)

In [6]:
# 5. Merge & clean sequences
df_all = pd.concat([df_go, df_sim1, df_sim2], ignore_index=True)
df_all.dropna(subset=['sequence','label'], inplace=True)

VALID_AA = set('ACDEFGHIKLMNPQRSTVWY')
df_all['sequence'] = df_all['sequence'].apply(
    lambda s: ''.join([aa if aa in VALID_AA else 'X' for aa in str(s).upper()])
)
df_all = df_all[df_all['sequence'].str.len().between(10, 1000)]
df_all.drop_duplicates(subset='sequence', inplace=True)

# Keep classes with ≥5 samples
vc = df_all['label'].value_counts()
df_all = df_all[df_all['label'].isin(vc[vc >= 5].index)]

print("Final shape:", df_all.shape)
print("Label distribution:")
print(df_all['label'].value_counts())

Final shape: (16158, 2)
Label distribution:
label
enzima         3235
estrutural     3232
transporte     3225
outras         3183
receptora      3125
binding         103
enzyme           31
other            16
transporter       8
Name: count, dtype: int64


In [7]:
# 6. Save processed sequences (for CNN) to data/processed/
df_all.to_csv(PROC_DIR / 'processed_sequences.csv', index=False)

# Save max sequence length (95th percentile) for padding
max_len = int(np.percentile(df_all['sequence'].apply(len), 95))
np.save(PROC_DIR / 'max_len.npy', max_len)
print("Max length for padding:", max_len)

Max length for padding: 290


In [8]:
# 7. Feature engineering for classical models (save to data/processed/)
AA_ORDER = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa:i for i,aa in enumerate(AA_ORDER)}

def aac(seq):
    counts = np.zeros(20)
    for aa in seq:
        if aa in AA_TO_IDX:
            counts[AA_TO_IDX[aa]] += 1
    return counts / len(seq) if len(seq)>0 else counts

def dpc(seq):
    vec = np.zeros(400)
    if len(seq)<2: return vec
    for i in range(len(seq)-1):
        di = seq[i:i+2]
        if di[0] in AA_TO_IDX and di[1] in AA_TO_IDX:
            idx = AA_TO_IDX[di[0]]*20 + AA_TO_IDX[di[1]]
            vec[idx] += 1
    return vec / (len(seq)-1)

X_aac = np.array([aac(seq) for seq in df_all['sequence']])
X_dpc = np.array([dpc(seq) for seq in df_all['sequence']])

# Encode labels
le = LabelEncoder()
y = le.fit_transform(df_all['label'])
class_names = le.classes_

# Save features
np.savez(PROC_DIR / 'features.npz', X_aac=X_aac, X_dpc=X_dpc, y=y)
np.save(PROC_DIR / 'class_names.npy', class_names)
print("Features saved: AAC", X_aac.shape, "DPC", X_dpc.shape, "classes", len(class_names))

Features saved: AAC (16158, 20) DPC (16158, 400) classes 9


---
**Data preparation complete.** All processed artifacts are now in `data/processed/`.